## ExPECA Testbed HI Setup

Start with reading README to understand the purpose of this notebook and the setup options.

---

### Overview

**The notebook will create a setup with the following components**

- Edge Device: Start a containerized application that runs the edge device in the ExPECA testbed.
  - Worker = Raspberry Pi (arm64).
- Edge Server: Start a containerized application that runs the edge server in the ExPECA testbed.
  - Worker = Server node (amd64).
- Communication: Connects the edge device to edge server wirelessly.
  - Ericsson private 5G network.


**Steps to follow:**

1. Set variables for the setup.
2. Set authentication for the ExPECA testbed.
3. Install required packages and libraries.
4. Reserve testbed resources.
5. Create networks and routers.
6. Setup Edge Server.
7. Setup Edge Device.

**Additional guides:**
- Starting the experiment.
- Collecting results and logs.

### Set Variables for the Setup

---

- Project name
- Advantech router
- Edge server worker
- Edge device worker

In [ ]:
project = "project_name"
adv_name = "adv-01"
server_worker_name = "worker-05" # 12 Core, 32 GB RAM
device_worker_name = "worker-21" # Raspberry Pi 4, 4 Core, 8 GB RAM

### Set Authentication for the ExPECA Testbed

---

**Steps:**

1. Login to Chameleon and download openrc.sh file from [here](https://testbed.expeca.proj.kth.se/project/api_access/openrc/).
2. Upload it here next to this notebook and continue.
3. Edit path to match your "xxx-project-openrc.sh" file if needed.
4. Enter your ExPECA password when asked.

In [ ]:
import os, re
from getpass import getpass

# Replace with your OpenStack credentials file path
with open('project_name-openrc.sh', 'r') as f:
    script_content = f.read()
    pattern = r'export\s+(\w+)\s*=\s*("[^"]+"|[^"\n]+)'
    matches = re.findall(pattern, script_content)

    for name, value in matches:
        os.environ[name] = value.strip('"')

password = getpass('enter your expeca password:')
os.environ['OS_PASSWORD'] = password

### Install Required Packages and Libraries

---

There might be some warnings, ignore them and test if everything still works.

In [ ]:
# Install required packages
!pip uninstall -q -y moviepy
!pip install -q jedi
!pip install -q git+https://github.com/KTH-EXPECA/python-chi

# Import required libraries
import json, time
from loguru import logger
import chi.network, chi.container, chi.network
from chi.expeca import reserve, list_reservations, unreserve_byid, get_container_status, wait_until_container_removed, show_reservation_byname, restart_sdr, make_sdr_ni, make_sdr_mango, sdr_tools, get_available_publicips, get_segment_ids, get_radio_interfaces, get_worker_interfaces

### Reserve Testbed Resources

---

Reserve the following testbed equipment based on the set variables:
- Edge Server
- Edge Device
- Advantech router
- Ericsson Private 5G


In [ ]:
# reserve workers
device_worker_lease = show_reservation_byname(device_worker_name+"-lease")
if not device_worker_lease:
    device_worker_lease = reserve(
        { "type":"device", "name":device_worker_name, "duration": { "days":7, "hours":0 } }
    )
device_worker_reservation_id = device_worker_lease["reservations"][0]["id"]

server_worker_lease = show_reservation_byname(server_worker_name+"-lease")
if not server_worker_lease:
    server_worker_lease = reserve(
        { "type":"device", "name":server_worker_name, "duration": { "days":7, "hours":0 } }
    )
server_worker_reservation_id = server_worker_lease["reservations"][0]["id"]

# reserve advantech router
segment_ids = get_segment_ids(adv_name)
adv_lease = show_reservation_byname(adv_name + "-lease")
if not adv_lease:
    adv_lease = reserve(
        { "type":"network", "name": adv_name, "net_name": adv_name, "segment_id": segment_ids['rj45'], "duration": { "days":7, "hours":0 } }
    )

# ep5g reservation
ep5g_lease = show_reservation_byname("ep5g-lease")
segment_ids = get_segment_ids('ep5g')
if not ep5g_lease:
    ep5g_lease = reserve(
        { "type":"network", "name": "ep5g", "net_name": "ep5g-vip", "segment_id": segment_ids['rj45'], "duration": { "days":7, "hours":0 } }
    )

leaseslist = list_reservations(brief=True)
print(json.dumps(leaseslist,indent=4))

### Create Networks and Routers

---
1. Create edge-net
2. Create ep5g-vip-router
3. Connect ep5g-vip-net to ep5g-vip-router
4. Connect edge-net to ep5g-vip-router
5. Add ep5g route to ep5g-vip-router

In [ ]:
# create edge-net
edgenet = chi.network.create_network("edge-net")
chi.network.create_subnet("edge-net-subnet",edgenet["id"],"10.70.70.0/24",gateway_ip=None,enable_dhcp=False)
logger.success("edge-net is created.")

# create ep5g-vip-router
router = chi.network.create_router("ep5g-vip-router", "public")
logger.success("ep5g-vip-router router is created.")
logger.info(f"{json.dumps(router,indent=4)}")

# connect ep5g-vip-net to ep5g-vip-router
ep5gnet = chi.network.get_network("ep5g-vip-net")
portadd = chi.network.add_subnet_to_router(router["id"],ep5gnet["subnets"][0])
logger.success("An interface on ep5g-vip-net is added to the router")

# connect edge-net to ep5g-vip-router
edgenet = chi.network.get_network("edge-net")
portadd = chi.network.add_subnet_to_router(router["id"],edgenet["subnets"][0])
logger.success("An interface on edge-net is added to the router")

# add ep5g route to ep5g-vip-router
routeadd = chi.network.add_route_to_router(router["id"],"172.16.0.0/16","10.30.111.10")
logger.success("the ep5g route added to the router")

### Setup Edge Server

---

1. Check public IPs and select the next available one.
2. Check available interfaces of the worker.
3. Load and run container.

In [ ]:
# 1. Check public IPs and select one
available_pub_ips = get_available_publicips()
if len(available_pub_ips) == 0:
  raise Exception("There is no available public IPs to reserve.")
pub_ip = available_pub_ips[0]
logger.info(f"Available public ips: {available_pub_ips}.")
logger.info(f"We choose {pub_ip} for this container.")

# 2. Check available interfaces on the worker
interfaces = list(get_worker_interfaces(server_worker_name).values())[0]
available_ifs = []
for interface in interfaces.keys():
  if len(interfaces[interface]['connections']) == 0:
    available_ifs.append(interface)
logger.info(f"Available interfaces on {server_worker_name}: {available_ifs}")

# 3. Load and run container
edgenet = chi.network.get_network("edge-net")
publicnet = chi.network.get_network("serverpublic")
container_name = "hi-edge-server"
chi.container.create_container(
    name = container_name,
    image = "h3nkk44/hi-framework-edge-server:latest_amd64_012",
    reservation_id = server_worker_reservation_id,
    environment = {
        "DNS_IP":"8.8.8.8",
        "GATEWAY_IP":"130.237.11.97",
        "PASS":"expeca"
    },
    mounts = [],
    nets = [
        { "network" : publicnet['id'] },
        { "network" : edgenet['id'] },
    ],
    labels = {
        "networks.1.interface":available_ifs[0],
        "networks.1.ip":pub_ip+"/27",
        "networks.1.gateway":"130.237.11.97",
        "capabilities.privileged":"true",
        "networks.2.interface":available_ifs[1],
        "networks.2.ip":"10.70.70.50/24",
        "networks.2.routes":"172.16.0.0/16-10.70.70.1",
    },
)
chi.container.wait_for_active(container_name)
logger.success(f"created {container_name} container, reachable at {pub_ip}.")

### Setup Edge Device

*NOTE: This is using arm64 image*
- For amd64 change to suitable worker and image with tag amd64.

---
1. Check available interfaces of the worker.
2. Load and run the container.

In [ ]:
# 1. Check available interfaces of the worker
interfaces = list(get_worker_interfaces(device_worker_name).values())[0]
available_ifs = []
for interface in interfaces.keys():
  if len(interfaces[interface]['connections']) == 0:
    available_ifs.append(interface)
logger.info(f"Available interfaces on {device_worker_name}: {available_ifs}")
if len(available_ifs) < 1:
  logger.info(f"{json.dumps(interfaces, indent=4)}")
  raise Exception(f"Did not find enough interfaces on {device_worker_name}")

# 2. Load and run container
advnet = chi.network.get_network(adv_name+"-net")
container_name = "hi-edge-device"
chi.container.create_container(
    name = container_name,
    image = "h3nkk44/hi-framework-edge-device:latest_arm64_012",
    reservation_id = device_worker_reservation_id,
    environment = {
        "DNS_IP":"8.8.8.8",
        "GATEWAY_IP":"10.42.3.1",
        "PASS":"expeca",
        "EDGE_SERVER_IP": "10.70.70.50"
    },
    nets = [
        { "network" : advnet['id'] },
    ],
    labels = {
        "capabilities.privileged":"true",
        "networks.1.interface":available_ifs[0],
        "networks.1.ip":"10.42.3.2/24",
        "networks.1.routes":"0.0.0.0/0-10.42.3.1",
    },
)
chi.container.wait_for_active(container_name)
logger.success(f"created {container_name} container.")

### Additional Guides

---

**1. Starting the Experiment:**

1. Open ExPECA Web GUI
2. Open Edge Device container console
 - Check if folder exist or old results (should be automatically deleted):
```bash
ls /app/results/
```
 - If not run:
```bash
rm -rf /app/results/*
```

3. Open Edge Server container console
 - Check if folder exist or old results (should be automatically deleted):
```bash
ls /app/results/
```
 - If not run:
```bash
rm -rf /app/results/*
```

4. Start experiment script with:
```bash	
/app/start.sh
```

5. Enter wanted datasets path (data/datasets/...):
```bash
imagenetV2/matched-frequency-format-val/
imagenetV2/test_subset/
imagenette/val_renamed/
```

6. Select other parameters to be used for the experiment
7. Don't close the console or web GUI or it will stop the experiment.
 - The console will show experiment finished message.

**2. Collecting Results and Logs:**

1. Open ExPECA Web GUI
2. Open Edge Device console
3. Insert command (set correct Edge Server public IP):
```bash
scp -v -r /app/results/ root@130.237.11.119:/app/
```

4. Open terminal on your local machine
5. Insert command (set correct Edge Server public IP):
```bash
scp -v -r root@130.237.11.119:/app/results/ C:/Users/henri/Downloads/run_001
```